# Week 1: Empirical Complexity & Big-O Benchmarking

**Z5007 Programming and Data Structures (MTech DSAI Core) · IIT Madras Zanzibar**

Step-counted comparisons, log-log growth exponent estimation, and Python `list` vs. `set` scaling.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z5007/week-01/labs/W01_Empirical_Complexity_BigO.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every algorithm is implemented from first principles in Python and verified with assertions.

## 1. Step-Counted Primitives across Asymptotic Classes

We instrument five operations to count elementary comparisons directly: `O(1)` array indexing, `O(log n)` binary search, `O(n)` linear scan, `O(n log n)` merge sort, and `O(n^2)` pairwise inversion counting.

In [1]:
import math
import random
import time


def constant_lookup(arr, index):
    return arr[index], 1


def binary_search_counted(sorted_arr, target):
    lo, hi = 0, len(sorted_arr) - 1
    steps = 0
    while lo <= hi:
        steps += 1
        mid = (lo + hi) // 2
        if sorted_arr[mid] == target:
            return mid, steps
        if sorted_arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1, steps


def linear_max_counted(arr):
    best = arr[0]
    steps = 0
    for val in arr[1:]:
        steps += 1
        if val > best:
            best = val
    return best, steps


def mergesort_counted(arr):
    if len(arr) <= 1:
        return arr[:], 0
    mid = len(arr) // 2
    left, c1 = mergesort_counted(arr[:mid])
    right, c2 = mergesort_counted(arr[mid:])
    out, i, j, comps = [], 0, 0, c1 + c2
    while i < len(left) and j < len(right):
        comps += 1
        if left[i] <= right[j]:
            out.append(left[i]); i += 1
        else:
            out.append(right[j]); j += 1
    return out + left[i:] + right[j:], comps


def count_inversions_quadratic(arr):
    invs, steps = 0, 0
    n = len(arr)
    for i in range(n):
        for j in range(i + 1, n):
            steps += 1
            if arr[i] > arr[j]:
                invs += 1
    return invs, steps


print(f"{'n':>7} | {'O(1)':>6} | {'O(log n)':>9} | {'O(n)':>8} | {'O(n log n)':>11} | {'O(n^2)':>10}")
print("-" * 64)
rng = random.Random(5007)
for n in (32, 128, 512, 1024):
    s_arr = list(range(n))
    r_arr = [rng.randint(0, 10_000) for _ in range(n)]
    _, c1 = constant_lookup(s_arr, n // 2)
    _, clog = binary_search_counted(s_arr, n - 1)
    _, clin = linear_max_counted(r_arr)
    _, cnlogn = mergesort_counted(r_arr)
    _, cquad = count_inversions_quadratic(r_arr)
    print(f"{n:>7} | {c1:>6} | {clog:>9} | {clin:>8} | {cnlogn:>11} | {cquad:>10}")


      n |   O(1) |  O(log n) |     O(n) |  O(n log n) |     O(n^2)
----------------------------------------------------------------
     32 |      1 |         6 |       31 |         122 |        496
    128 |      1 |         8 |      127 |         734 |       8128
    512 |      1 |        10 |      511 |        3975 |     130816
   1024 |      1 |        11 |     1023 |        8974 |     523776


## 2. Empirical Log-Log Exponent Estimation & Python `list` vs. `set` Membership

Taking logarithms of $T(n) \approx c n^b$ yields $\log T(n) \approx \log c + b \log n$, so the slope of a least-squares line in log-log coordinates recovers the empirical exponent $b$. We also contrast `O(n)` list scan against `O(1)` hash-set membership.

In [2]:
def estimate_log_log_slope(sizes, times_sec):
    xs = [math.log(n) for n in sizes]
    ys = [math.log(max(t, 1e-12)) for t in times_sec]
    mx, my = sum(xs) / len(xs), sum(ys) / len(ys)
    num = sum((x - mx) * (y - my) for x, y in zip(xs, ys))
    den = sum((x - mx) ** 2 for x in xs)
    return num / den


sizes = [150, 300, 600, 1200]
quad_times = []
for n in sizes:
    data = [rng.randint(0, 10_000) for _ in range(n)]
    t0 = time.perf_counter()
    count_inversions_quadratic(data)
    quad_times.append(time.perf_counter() - t0)

slope = estimate_log_log_slope(sizes, quad_times)
print(f"Estimated quadratic exponent b = {slope:.2f} (theoretical = 2.00)")
assert 1.5 <= slope <= 2.5

# List vs Set membership on 50,000 elements
big_list = list(range(50_000))
big_set = set(big_list)
queries = [-1, 49_999, 25_000, 99_999] * 50

t0 = time.perf_counter()
list_hits = sum(1 for q in queries if q in big_list)
t_list = time.perf_counter() - t0

t0 = time.perf_counter()
set_hits = sum(1 for q in queries if q in big_set)
t_set = time.perf_counter() - t0

assert list_hits == set_hits
print(f"200 lookups on 50k items -> list: {t_list*1000:.2f} ms, set: {t_set*1000:.4f} ms")


Estimated quadratic exponent b = 2.14 (theoretical = 2.00)
200 lookups on 50k items -> list: 188.70 ms, set: 0.0278 ms
